# Mushuk va It klassifikatori — ML jarayonini amalda ko'rish

Bu notebook — darsimizda gaplashgan **Ma'lumot → Train → Test → Evaluate** jarayonining haqiqiy kod bilan bajarilishi.

Biz kompyuterga rasmga qarab **"bu mushukmi yoki itmi?"** ni aniqlashni o'rgatamiz. Har bir qadamda **nima qilinayotgani** va **nega aynan shunday qilinayotgani** tushuntiriladi.

**Ma'lumotlar haqida:** bu notebook hech qanday tashqi papka yoki o'zingiz tayyorlagan rasm talab qilmaydi — **1-qadamda ma'lumot avtomatik yuklab olinadi** (mashhur, ochiq "Cats vs Dogs" to'plamidan kichik bir qism). Faylni istalgan joyga qo'yib, hujayralarni tartib bilan ishga tushirsangiz bo'ldi.

## 1-qadam: Kerakli kutubxonalarni yuklash

Kod yozishdan oldin, bizga kerak bo'ladigan "vositalar"ni chaqirib olamiz:

- **TensorFlow / Keras** — modelni qurish va o'qitish uchun asosiy kutubxona
- **matplotlib** — rasmlarni va grafiklarni ko'rsatish (vizualizatsiya) uchun
- **numpy** — sonlar bilan ishlash uchun
- **scikit-learn** — natijalarni baholash (accuracy, confusion matrix) uchun

*Bu — xuddi oshxonaga kirishdan oldin kerakli asboblarni stolga qo'yib chiqishga o'xshaydi.*

In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow versiyasi:", tf.__version__)


## 2-qadam: Ma'lumotlarni tayyorlash (Data loading)

Bu — darsimizdagi **birinchi bosqich: Ma'lumot to'plash**.

**Muhim:** internetdan real rasm yuklab olish ba'zi tarmoqlarda (maktab/idora Wi-Fi'i, firewall, antivirus) sekin yoki barqarorsiz ishlashi mumkin — bu sizning aybingiz emas, tarmoq cheklovlari tufayli. Shuning uchun bu notebook **standart holatda internetga umuman muhtoj bo'lmaydi**: mushuk va it rasmlarini **o'zi chizib chiqaradi** (oddiy, sxematik, lekin aniq ikki xil ko'rinishdagi "yuzlar" — uchli quloqli mushuk, osilib turgan quloqli it). Bu — 100% barqaror ishlaydi, video yozishda hech qanday kutish yoki xatolik bo'lmaydi.

Agar internetingiz barqaror bo'lsa va haqiqiy fotosurat bilan ishlashni xohlasangiz, pastdagi `USE_REAL_PHOTOS = True` qilib o'zgartirsangiz bo'ladi (Wikimedia Commons'dan yuklaydi) — lekin video yozish uchun **False (standart)** qoldirishni tavsiya qilamiz.

In [ ]:
# Sozlama: standart holatda offline (chizilgan) rasm ishlatiladi — 100% barqaror
USE_REAL_PHOTOS = False   # True qilsangiz, Wikimedia'dan haqiqiy foto yuklashga urinadi (internet talab qiladi)
N_PER_CLASS = 25          # har klassdan nechta rasm; oshirsangiz train biroz uzoqroq davom etadi
SUBSET_DIR = "cats_vs_dogs_subset"
IMG_CANVAS = 160          # chizilgan rasm o'lchami (piksel)


### 2.1 — Offline rasm generatori (standart, internet kerak emas)

Quyidagi funksiyalar `PIL` (rasm bilan ishlash kutubxonasi) yordamida oddiy "mushuk yuzi" va "it yuzi" chizadi — har safar ozgina tasodifiy farq bilan (rang, o'lcham, joylashuv), xuddi haqiqiy rasmlar kabi xilma-xil bo'lishi uchun.

**Nega bu yetarli?** Bizning maqsadimiz — aniq fotosurat klassifikatori emas, balki **ML jarayonini** (train/test/evaluate) tushunish. Ikki aniq farqli shakl (uchli quloq vs osilgan quloq) buning uchun to'liq yetarli.

In [ ]:
from PIL import Image, ImageDraw
import random

def _rand_color(lo=110, hi=230):
    return tuple(int(x) for x in np.random.randint(lo, hi, 3))

def draw_cat(size=IMG_CANVAS):
    bg = _rand_color(215, 250)
    img = Image.new("RGB", (size, size), bg)
    d = ImageDraw.Draw(img)
    cx, cy = size // 2 + random.randint(-8, 8), size // 2 + random.randint(-8, 8)
    r = random.randint(int(size * 0.26), int(size * 0.34))
    fur = _rand_color(120, 220)
    d.polygon([(cx - r * 0.9, cy - r * 0.4), (cx - r * 0.3, cy - r * 1.5), (cx - r * 0.1, cy - r * 0.5)], fill=fur)
    d.polygon([(cx + r * 0.9, cy - r * 0.4), (cx + r * 0.3, cy - r * 1.5), (cx + r * 0.1, cy - r * 0.5)], fill=fur)
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=fur)
    eye_off = r * 0.4
    for sx in (-1, 1):
        ex = cx + sx * eye_off
        d.ellipse([ex - 6, cy - 6, ex + 6, cy + 6], fill="black")
    for dy in (-8, 0, 8):
        d.line([(cx - r * 1.3, cy + 15 + dy), (cx - r * 0.5, cy + 10 + dy)], fill=(90, 90, 90), width=2)
        d.line([(cx + r * 0.5, cy + 10 + dy), (cx + r * 1.3, cy + 15 + dy)], fill=(90, 90, 90), width=2)
    d.polygon([(cx - 5, cy + 8), (cx + 5, cy + 8), (cx, cy + 16)], fill=(230, 150, 160))
    return img

def draw_dog(size=IMG_CANVAS):
    bg = _rand_color(215, 250)
    img = Image.new("RGB", (size, size), bg)
    d = ImageDraw.Draw(img)
    cx, cy = size // 2 + random.randint(-8, 8), size // 2 + random.randint(-8, 8)
    r = random.randint(int(size * 0.26), int(size * 0.34))
    fur = _rand_color(100, 200)
    d.ellipse([cx - r * 1.35, cy - r * 0.3, cx - r * 0.55, cy + r], fill=fur)
    d.ellipse([cx + r * 0.55, cy - r * 0.3, cx + r * 1.35, cy + r], fill=fur)
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=fur)
    eye_off = r * 0.4
    for sx in (-1, 1):
        ex = cx + sx * eye_off
        d.ellipse([ex - 6, cy - 6, ex + 6, cy + 6], fill="black")
    snout = tuple(min(c + 25, 255) for c in fur)
    d.ellipse([cx - r * 0.4, cy + r * 0.15, cx + r * 0.4, cy + r * 0.75], fill=snout)
    d.ellipse([cx - 7, cy + r * 0.25, cx + 7, cy + r * 0.25 + 14], fill="black")
    d.polygon([(cx - 7, cy + r * 0.65), (cx + 7, cy + r * 0.65), (cx, cy + r * 1.0)], fill=(230, 130, 150))
    return img

def generate_offline_dataset(n_per_class, dest_dir):
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    for cls, draw_fn in [("cats", draw_cat), ("dogs", draw_dog)]:
        cls_dir = os.path.join(dest_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)
        for i in range(n_per_class):
            img = draw_fn()
            img.save(os.path.join(cls_dir, f"{i}.png"))
    n_cats = len(os.listdir(os.path.join(dest_dir, "cats")))
    n_dogs = len(os.listdir(os.path.join(dest_dir, "dogs")))
    return n_cats, n_dogs


### 2.2 — (Ixtiyoriy) Haqiqiy fotosurat — Wikimedia Commons orqali

Bu qism faqat `USE_REAL_PHOTOS = True` bo'lsa ishga tushadi. Wikimedia Commons — ochiq litsenziyali rasmlar ombori; ba'zi tarmoqlarda sekinroq ishlashi mumkin, shuning uchun ixtiyoriy qilib qoldirildi.

In [ ]:
import urllib.request
import urllib.parse
import json

COMMONS_API = "https://commons.wikimedia.org/w/api.php"
HEADERS = {"User-Agent": "MLTeachingLab-Uzbekistan/1.0 (educational demo; ml-teaching-lab)"}
VALID_EXT = (".jpg", ".jpeg", ".png")

def _api_get(params):
    url = COMMONS_API + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=15) as r:
        return json.load(r)

def get_category_file_titles(category, limit=60):
    data = _api_get({
        "action": "query", "list": "categorymembers",
        "cmtitle": f"Category:{category}", "cmtype": "file",
        "cmlimit": str(limit), "format": "json",
    })
    return [m["title"] for m in data.get("query", {}).get("categorymembers", [])]

def get_direct_urls(titles):
    urls = []
    for i in range(0, len(titles), 50):
        batch = titles[i:i + 50]
        data = _api_get({
            "action": "query", "titles": "|".join(batch),
            "prop": "imageinfo", "iiprop": "url", "format": "json",
        })
        for page in data.get("query", {}).get("pages", {}).values():
            info = page.get("imageinfo")
            if info:
                url = info[0]["url"]
                if url.lower().endswith(VALID_EXT):
                    urls.append(url)
    return urls

def download_images(urls, dest_dir, n):
    os.makedirs(dest_dir, exist_ok=True)
    saved = 0
    for url in urls:
        if saved >= n:
            break
        try:
            ext = os.path.splitext(url)[1].lower()
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=10) as resp, \
                 open(os.path.join(dest_dir, f"{saved}{ext}"), "wb") as f:
                f.write(resp.read())
            saved += 1
        except Exception:
            continue
    return saved

def generate_online_dataset(n_per_class, dest_dir):
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    cat_urls = get_direct_urls(get_category_file_titles("Cats", limit=n_per_class * 4))
    dog_urls = get_direct_urls(get_category_file_titles("Dogs", limit=n_per_class * 4))
    n_cats = download_images(cat_urls, os.path.join(dest_dir, "cats"), n_per_class)
    n_dogs = download_images(dog_urls, os.path.join(dest_dir, "dogs"), n_per_class)
    return n_cats, n_dogs


### 2.3 — Ma'lumotni tayyorlash (tanlangan usul bo'yicha)

In [ ]:
if USE_REAL_PHOTOS:
    n_cats, n_dogs = generate_online_dataset(N_PER_CLASS, SUBSET_DIR)
    print(f"(Online) Mushuk rasmlari: {n_cats} | It rasmlari: {n_dogs}")
    if n_cats < 4 or n_dogs < 4:
        print("\nInternet orqali yetarli rasm yuklanmadi — offline generatorga o'tilmoqda...")
        n_cats, n_dogs = generate_offline_dataset(N_PER_CLASS, SUBSET_DIR)
        print(f"(Offline) Mushuk rasmlari: {n_cats} | It rasmlari: {n_dogs}")
else:
    n_cats, n_dogs = generate_offline_dataset(N_PER_CLASS, SUBSET_DIR)
    print(f"(Offline) Mushuk rasmlari: {n_cats} | It rasmlari: {n_dogs}")


### Endi tayyorlangan rasmlarni Train/Test'ga bo'lamiz

Bu yerda darsimizdagi **80/20 train/test split** amalga oshadi — `validation_split=0.2` degani ma'lumotning 80%i train, 20%i test (validation) uchun ajratiladi.

**Nega rasm o'lchamini bir xillashtiramiz (`image_size=(128,128)`)?** Chunki model doim bir xil "shakldagi" kirish kutadi — xuddi barcha o'quvchilar bir xil o'lchamdagi varaqqa imtihon yozishi kabi.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 8
SEED = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    SUBSET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    SUBSET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Klasslar:", class_names)
print("Klasslar mosligi -> 0 =", class_names[0], "| 1 =", class_names[1])


## 3-qadam: Ma'lumotni ko'zdan kechirish (vizualizatsiya)

**Muhim odat:** kodni davom ettirishdan oldin, ma'lumotingizga albatta bir marta ko'z bilan qarab chiqing. Bu xatolarni (masalan, noto'g'ri papkaga tushib qolgan rasm) oldindan aniqlashga yordam beradi.

Pastda train to'plamidan bir nechta namunani, ularning haqiqiy yorlig'i bilan birga ko'ramiz.

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(min(8, len(images))):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.suptitle("Train to'plamidan namunalar")
plt.tight_layout()
plt.show()


## 4-qadam: Ishlash tezligini optimallashtirish

Bu qadam modelning ishlashiga emas, balki **tezligiga** ta'sir qiladi — `cache()` ma'lumotni xotirada saqlaydi, `prefetch()` esa keyingi partiyani oldindan tayyorlab qo'yadi. Kichik dataset uchun katta farq bo'lmasa-da, bu — professional Keras loyihalarida standart amaliyot.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(50).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 5-qadam: Data augmentation — nega kerak?

Bizda har klassdan atigi **~25 tadan rasm** bor (tezkor demo uchun ataylab kichraytirilgan) — bu haqiqiy loyihalar uchun juda kam. Darsimizda aytganimizdek, kam ma'lumot modelni "yodlab qolish"ga (overfitting) olib kelishi mumkin.

**Data augmentation** — mavjud rasmlarni aylantirib, kattalashtirib, oyna kabi aks ettirib, sun'iy ravishda "xilma-xillik" qo'shish usuli. Bu modelga *"mushuk har doim bir xil burchakda turmaydi"* ekanini o'rgatishga yordam beradi.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


## 6-qadam: Modelni qurish — Transfer Learning

Bizda har klassdan atigi ~25 tadan rasm bor. Agar noldan katta neyron tarmoq qursak, u hech narsani o'rgana olmaydi — ma'lumot juda kam.

Shuning uchun **Transfer Learning** (ko'chirib o'rganish) usulidan foydalanamiz: **MobileNetV2** — millionlab rasmda (ImageNet to'plamida) allaqachon o'qitilgan tayyor modeldan foydalanamiz. Bu model shakllarni, chiziqlarni, teksturalarni tanishni allaqachon biladi.

*Bu — xuddi noldan chizishni o'rganish o'rniga, tajribali rassomdan asosiy texnikalarni o'rganib, keyin faqat o'zingizning mavzuingizga (mushuk/it) moslashtirishga o'xshaydi.*

`base_model.trainable = False` — bu qatorda biz MobileNetV2ning bilimini **"muzlatib qo'yamiz"** (o'zgartirmaymiz), faqat uning ustiga kichik, o'zimizning "mushuk vs it" qarorini beruvchi qismni qo'shamiz.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
)

model.summary()


## 7-qadam: Modelni o'qitish (Train)

Bu — darsimizdagi **ikkinchi bosqich: Train**.

- **epoch** — model butun train ma'lumotini nechchi marta "qayta ko'rib chiqishi". Har epoch'da model biroz yaxshilanadi.
- **validation_data** — har epoch oxirida modelni test to'plamida ham tekshirib turamiz, shunda train paytida ham "imtihon natijasi"ni kuzatib boramiz.

Dataset juda kichik bo'lgani uchun ko'p epoch shart emas — 10 ta epoch yetarli.

In [ ]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)


## 8-qadam: Train jarayonini vizualizatsiya qilish

Endi modelning epoch-epoch qanday o'rganganini grafikda ko'ramiz — accuracy (aniqlik) oshib borishi va loss (xato) kamayib borishi kerak.

**Diqqat qiling:** agar train accuracy juda yuqori bo'lib, val (test) accuracy past yoki beqaror bo'lsa — bu **overfitting** belgisi bo'lishi mumkin (darsda gaplashganimizdek, kichik dataset bilan bu tabiiy holat).

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Train accuracy")
plt.plot(epochs_range, val_acc, label="Test accuracy")
plt.legend(loc="lower right")
plt.title("Aniqlik (Accuracy)")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Train loss")
plt.plot(epochs_range, val_loss, label="Test loss")
plt.legend(loc="upper right")
plt.title("Xato (Loss)")

plt.show()


## 9-qadam: Test to'plamida baholash (Evaluate)

Bu — darsimizdagi **uchinchi va to'rtinchi bosqich: Test va Evaluate**.

Model endi **oldin ko'rmagan** rasmlarda (test to'plamida) sinaladi. Natijada chiqadigan **accuracy** — bizning asosiy baholash mezonimiz.

In [ ]:
test_loss, test_accuracy = model.evaluate(val_ds)
print(f"\nTest (baholash) aniqligi: {test_accuracy * 100:.1f}%")
print(f"Test xatoligi (loss): {test_loss:.3f}")


### Batafsil baholash: Confusion Matrix va Classification Report

Faqat umumiy accuracy'ni bilish yetarli emas — qaysi klassda ko'proq xato qilinganini ham ko'rish foydali. Buning uchun **confusion matrix** (chalkashlik matritsasi) va **classification report** dan foydalanamiz.

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    preds = (preds > 0.5).astype(int).flatten()
    y_true.extend(labels.numpy())
    y_pred.extend(preds)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Model bashorati")
plt.ylabel("Haqiqiy yorliq")
plt.title("Confusion Matrix")
plt.show()


## 10-qadam: Bashoratlarni vizual ko'rish

Eng tushunarli qism — modelning har bir test rasmiga bergan bashoratini, haqiqiy javob bilan yonma-yon ko'ramiz. **Yashil ramka** — to'g'ri topilgan, **qizil ramka** — xato.

Bu — darsda sizning simulyatoringizda ko'rgan ✓/✗ belgilarining, endi haqiqiy rasmlar bilan ko'rinishi.

In [ ]:
plt.figure(figsize=(12, 8))
i = 0
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    for img, true_label, pred in zip(images, labels, preds):
        if i >= 8:
            break
        pred_label = 1 if pred[0] > 0.5 else 0
        confidence = pred[0] if pred_label == 1 else 1 - pred[0]
        is_correct = pred_label == true_label.numpy()

        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(img.numpy().astype("uint8"))
        plt.axis("off")
        color = "green" if is_correct else "red"
        for spine in ["top", "bottom", "left", "right"]:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_color(color)
            ax.spines[spine].set_linewidth(4)
        plt.title(f"Haqiqiy: {class_names[true_label]}\nBashorat: {class_names[pred_label]} ({confidence*100:.0f}%)", fontsize=9)
        i += 1
    if i >= 8:
        break

plt.tight_layout()
plt.show()


## 10.1-qadam: Natijani tushunish — model haqiqatan ham "o'rgandimi"?

Ba'zida model faqat bitta klassni (masalan, doim "dogs") bashorat qilib qo'yishi mumkin — bunga **"dangasa model" (lazy / majority-class model)** deyiladi. Bu — real ML muhandisligida juda tez-tez uchraydigan holat, ayniqsa ma'lumot kam yoki klasslar soni teng bo'lmaganda.

Quyidagi kod avtomatik tekshiradi: model ikkala klassni ham bashorat qildimi, yoki faqat bittasini takrorladimi?

In [ ]:
from collections import Counter

pred_counts = Counter(y_pred)
true_counts = Counter(y_true)

print("Haqiqiy taqsimot (test to'plamida):", {class_names[k]: v for k, v in true_counts.items()})
print("Model bashoratlari taqsimoti:     ", {class_names[k]: v for k, v in pred_counts.items()})

if len(pred_counts) == 1:
    only_class = class_names[list(pred_counts.keys())[0]]
    majority_share = max(true_counts.values()) / sum(true_counts.values())
    print(f"\nDIQQAT: Model BARCHA test namunalarini '{only_class}' deb bashorat qildi.")
    print("Bu — 'dangasa model' holati: model ikkala klassni chindan farqlashni o'rganish o'rniga,")
    print(f"eng ko'p uchragan javobni takrorlab, tasodifan {majority_share*100:.0f}% aniqlikka erishdi.")
    print("Sabab, ehtimol: (1) juda kichik dataset, (2) chizilgan tekis rasmlar MobileNetV2 uchun")
    print("notanish (u haqiqiy fotosuratlarda o'qitilgan), (3) test to'plamidagi klasslar soni teng emas.")
else:
    print("\nModel ikkala klassni ham bashorat qildi — bu model haqiqatan ham farqlashga harakat")
    print("qilganini ko'rsatadi, garchi aniqlik yuqori bo'lmasa ham.")


### Bu darsda buni qanday tushuntirish mumkin

Bu natija — **kamchilik emas, balki juda qimmatli dars**. Sinfda buni shunday tushuntirish mumkin:

> *"Ko'rdingizmi, modelimiz unchalik aqlli ishlamadi — ba'zida u shunchaki 'eng ko'p uchragan javobni' takrorlayapti, xuddi imtihonda bilmagan savolga eng ko'p uchraydigan javobni belgilagandek. Bu — machine learning'da juda muhim va real muammo: agar ma'lumot kam bo'lsa yoki klasslar soni teng bo'lmasa, model 'aldash yo'lini' topib oladi. Aynan shuning uchun professional ML muhandislari nafaqat accuracy'ga, balki confusion matrix'ga ham qarashadi — chunki accuracy yolg'on ishonch berishi mumkin!"*

**Asosiy tushunchalar (o'quvchilarga ta'kidlash kerak bo'lgan fikrlar):**
- Yuqori accuracy har doim ham "model yaxshi ishlayapti" degani emas — buni darsimizning boshida ham gaplashgan edik
- Confusion matrix — modelning "yolg'on ishonchini" fosh qilishga yordam beradi
- Kam ma'lumot va klasslar nomutanosibligi — real loyihalarda ham eng ko'p uchraydigan muammolardan biri
- Bu — modelning "yomon" ekanini emas, balki **to'g'ri ma'lumot va sinov muhim ekanini** ko'rsatadi

## Xulosa

Ushbu notebookda biz darsimizdagi to'rtta bosqichni **haqiqiy kod** bilan bajardik:

| Bosqich | Bu yerda qanday amalga oshirildi |
|---|---|
| **Ma'lumot** | Standart holatda avtomatik chizilgan (offline) rasmlar, har klassdan `N_PER_CLASS` (standart: 25) ta; xohlasangiz haqiqiy fotosuratga o'tish mumkin |
| **Train** | Model 80% ma'lumot asosida o'qitildi, Transfer Learning (MobileNetV2) yordamida |
| **Test** | Model qolgan 20% — oldin ko'rmagan ma'lumotda sinaldi |
| **Evaluate** | Accuracy, confusion matrix, classification report va "dangasa model" tekshiruvi orqali baholandi |

**Muhim eslatma (darsda aytish uchun):** biz ataylab kichik va soddalashtirilgan ma'lumot (`N_PER_CLASS=25`, chizilgan rasmlar) ishlatdik — tezkor va barqaror demo uchun. Real loyihalarda odatda minglab, haqiqiy fotosurat ishlatiladi. Shuning uchun bu yerdagi natijalar 100% barqaror yoki yuqori bo'lmasligi mumkin — **va bu ham darsimizdagi eng muhim tushunchani jonli tasdiqlaydi**: kam va sun'iy ma'lumot ishonchsiz (hatto "dangasa") natijaga olib kelishi mumkin. Agar model bitta klassni takrorlab qo'ysa, 10.1-qadamdagi tahlil buni aniq ko'rsatadi va tushuntiradi.

**Keyingi qadam sifatida sinab ko'rish mumkin:**
- Har klass uchun ko'proq rasm qo'shish
- `base_model.trainable = True` qilib, MobileNetV2ni ham "fine-tune" qilish (ilg'or mavzu)
- Boshqa hayvon turlarini qo'shib, ko'p klassli klassifikatorga aylantirish
